In [ ]:
!pip install --upgrade pip awscli botocore boto3  --quiet

In [ ]:
!pip install tensorflow==2.18

In [ ]:
from sagemaker import Session, image_uris
import boto3
import time

In [ ]:
region = 'us-east-1'
role = 'AmazonSageMaker-ExecutionRole-20241108T131283'
sm_client = boto3.client("sagemaker", region_name=region)
sagemaker_session = Session()

In [ ]:
import tensorflow as tf

# ML framework details
framework = "tensorflow"
# Note that only the framework major and minor version is supported for Neo compilation
framework_version = ".".join(tf.__version__.split(".")[:-1])

# model name as standardized by model zoos or a similar open source model
model_name = "resnet50"

# ML model details
ml_domain = "COMPUTER_VISION"
ml_task = "IMAGE_CLASSIFICATION"

print("TF Version", framework_version)

In [ ]:
download_the_model = True

In [ ]:
import os
import tensorflow as tf

In [ ]:
import tensorflow as tf

if download_the_model:
    input_tensor = tf.keras.Input(name="input_1", shape=(224, 224, 3))
    model = tf.keras.applications.resnet50.ResNet50(input_tensor=input_tensor)

    # Creating the directory structure
    model_version = "1"
    export_dir = "./model/" + model_version
    if not os.path.exists(export_dir):
        os.makedirs(export_dir)
        print("Directory ", export_dir, " Created ")
    else:
        print("Directory ", export_dir, " already exists")

    # Export to SavedModel
    model.export(export_dir)

In [ ]:
os.makedirs("code")

In [ ]:
%%writefile code/inference.py

import io
import json
import numpy as np
from PIL import Image

IMAGE_SIZE = (224, 224)


def input_handler(data, context):
    """Pre-process request input before it is sent to TensorFlow Serving REST API
    https://github.com/aws/amazon-sagemaker-[REDACTED]acb8d62e3b3409/sagemaker-python-sdk/tensorflow_serving_container/sample_utils.py#L61

    Args:
        data (obj): the request data stream
        context (Context): an object containing request and configuration details

    Returns:
        (dict): a JSON-serializable dict that contains request body and headers
    """

    if context.request_content_type == "application/x-image":
        buf = np.fromstring(data.read(), np.uint8)
        image = Image.open(io.BytesIO(buf)).resize(IMAGE_SIZE)
        image = np.array(image)
        image = np.expand_dims(image, axis=0)
        return json.dumps({"instances": image.tolist()})
    else:
        _return_error(
            415, 'Unsupported content type "{}"'.format(context.request_content_type or "Unknown")
        )


def output_handler(response, context):
    """Post-process TensorFlow Serving output before it is returned to the client.

    Args:
        response (obj): the TensorFlow serving response
        context (Context): an object containing request and configuration details

    Returns:
        (bytes, string): data to return to client, response content type
    """
    if response.status_code != 200:
        _return_error(response.status_code, response.content.decode("utf-8"))
    response_content_type = context.accept_header
    prediction = response.content
    return prediction, response_content_type


def _return_error(code, message):
    raise ValueError("Error: {}, {}".format(str(code), message))

In [ ]:
# Import necessary modules
import io
from PIL import Image

# Assuming inference.py is already written to code/inference.py
from inference_script import input_handler, output_handler

# Test input_handler
# Create a sample image
image = Image.new('RGB', (224, 224), color='red')
buf = io.BytesIO()
image.save(buf, format='JPEG')
buf.seek(0)

# Mock context for input_handler
class MockContext:
    def __init__(self):
        self.request_content_type = "application/x-image"
        self.accept_header = "application/json"

context = MockContext()

# Call input_handler with valid content type
try:
    input_result = input_handler(buf, context)
    print("Input Handler Result (valid):", input_result)
except ValueError as e:
    print("Input Handler Error (valid):", e)

# Test input_handler with wrong content type
wrong_context = MockContext()
wrong_context.request_content_type = "text/plain"

try:
    input_handler(buf, wrong_context)
except ValueError as e:
    print("Input Handler Error (wrong content type):", e)

# Test output_handler
# Mock response for output_handler
class MockResponse:
    def __init__(self, status_code, content):
        self.status_code = status_code
        self.content = content

# Successful response
mock_response = MockResponse(200, b'{"predictions": [1, 2, 3]}')

try:
    output_result, content_type = output_handler(mock_response, context)
    print("Output Handler Result (success):", output_result.decode('utf-8'))
    print("Output Handler Content Type (success):", content_type)
except ValueError as e:
    print("Output Handler Error (success):", e)

# Error response
error_response = MockResponse(400, b"Bad Request")

try:
    output_handler(error_response, context)
except ValueError as e:
    print("Output Handler Error (error status):", e)

In [ ]:
!pip freeze

In [ ]:
%%writefile code/requirements.txt

numpy==1.26.4
pillow==11.2.1

In [ ]:
model_archive_name = "tfmodel.tar.gz"

In [ ]:
!tar -cvpzf {model_archive_name} ./model ./code

In [ ]:
# model package tarball (model artifact + inference code)
model_url = sagemaker_session.upload_data(path=model_archive_name, key_prefix="tfmodel")
print("model uploaded to: {}".format(model_url))

In [ ]:
payload_archive_name = "tf_payload.tar.gz"

In [ ]:
## optional: download sample images
SAMPLES_BUCKET = f"sagemaker-example-files-prod-{region}"
PREFIX = "datasets/image/pets/"
payload_location = "./sample-payload/"

if not os.path.exists(payload_location):
    os.makedirs(payload_location)
    print("Directory ", payload_location, " Created ")
else:
    print("Directory ", payload_location, " already exists")

sagemaker_session.download_data(payload_location, SAMPLES_BUCKET, PREFIX)

In [ ]:
!cd ./sample-payload/ && tar czvf ../{payload_archive_name} *

In [ ]:
sample_payload_url = sagemaker_session.upload_data(
    path=payload_archive_name, key_prefix="tf_payload"
)

In [ ]:
print(sample_payload_url)

In [ ]:
instance_type = "ml.c5.xlarge"  # Note: you can use any CPU-based instance type here, this is just to get a CPU tagged image
dlc_uri = image_uris.retrieve(
    framework,
    region,
    version=framework_version,
    py_version="py3",
    instance_type=instance_type,
    image_scope="inference",
)
dlc_uri

In [ ]:
!pip install --upgrade sagemaker

In [ ]:
model_package_group_name = "{}-cpu-models-".format(framework) + str(round(time.time()))
model_package_group_description = "{} models".format(ml_task.lower())

model_package_group_input_dict = {
    "ModelPackageGroupName": model_package_group_name,
    "ModelPackageGroupDescription": model_package_group_description,
}

create_model_package_group_response = sm_client.create_model_package_group(
    **model_package_group_input_dict
)
print(
    "ModelPackageGroup Arn : {}".format(create_model_package_group_response["ModelPackageGroupArn"])
)

In [ ]:
model_package_description = "{} {} inference recommender".format(framework, model_name)

model_approval_status = "PendingManualApproval"

create_model_package_input_dict = {
    "ModelPackageGroupName": model_package_group_name,
    "Domain": ml_domain.upper(),
    "Task": ml_task.upper(),
    "SamplePayloadUrl": sample_payload_url,
    "ModelPackageDescription": model_package_description,
    "ModelApprovalStatus": model_approval_status,
}

In [ ]:
input_mime_types = ["application/x-image"]

In [ ]:
supported_realtime_inference_types = ["ml.c5.xlarge", "ml.m5.large", "ml.inf1.xlarge"]

In [ ]:
!saved_model_cli show --dir {export_dir} --all

In [ ]:
data_input_configuration = '{"input_1":[1,224,224,3]}'

In [ ]:
modelpackage_inference_specification = {
    "InferenceSpecification": {
        "Containers": [
            {
                "Image": dlc_uri,
                "Framework": framework.upper(),
                "FrameworkVersion": framework_version,
                "NearestModelName": model_name,
                "ModelInput": {"DataInputConfig": data_input_configuration},
            }
        ],
        "SupportedContentTypes": input_mime_types,  # required, must be non-null
        "SupportedResponseMIMETypes": [],
        "SupportedRealtimeInferenceInstanceTypes": supported_realtime_inference_types,  # optional
    }
}

# Specify the model data
modelpackage_inference_specification["InferenceSpecification"]["Containers"][0][
    "ModelDataUrl"
] = model_url

In [ ]:
create_model_package_input_dict.update(modelpackage_inference_specification)

In [ ]:
create_mode_package_response = sm_client.create_model_package(**create_model_package_input_dict)
model_package_arn = create_mode_package_response["ModelPackageArn"]
print("ModelPackage Version ARN : {}".format(model_package_arn))

In [ ]:
sm_client.describe_model_package(ModelPackageName=model_package_arn)

In [ ]:
inference_client = boto3.client("sagemaker", region)
list_job_steps_response = inference_client.list_inference_recommendations_job_steps(
    JobName='inference-recommender-job-1749766883632'
)
print(list_job_steps_response)

In [ ]:
role = 'arn:aws:iam::794038231401:role/service-role/SageMaker-ExecutionRole-20250401T103257'

In [ ]:
import boto3
import uuid

inference_client = boto3.client('sagemaker',region)

role = role
advanced_job = uuid.uuid1()

advanced_response = inference_client.create_inference_recommendations_job(
    JobName = str(advanced_job),
    JobDescription='',
    JobType = 'Advanced',
    RoleArn = role,
    InputConfig = {
        'ModelPackageVersionArn': model_package_arn,
        'JobDurationInSeconds': 7200,
        'EndpointConfigurations': [
            {
                'InstanceType': instance_type,
                'EnvironmentParameterRanges': {
                    'CategoricalParameterRanges': [
                        { 'Name': "OMP_NUM_THREADS","Value": ["1",'2','4']}
                    ]
                },
            }
        ],
        'ResourceLimit': {'MaxNumberOfTests': 3, 'MaxParallelOfTests': 1},
        'TrafficPattern': {
            'TrafficType': 'PHASES',
            'Phases': [{'InitialNumberOfUsers':1, 'SpawnRate': 1, 'DurationInSeconds': 120}],
        }
    },
    StoppingConditions = {
        'MaxInvocations': 1000,
        'ModelLatencyThresholds': [{'Percentile': 'P95', 'ValueInMilliseconds': 500}]
    },
)
print(advanced_response)

In [ ]:
import boto3
import uuid
import pandas as pd

inference_client = boto3.client("sagemaker", region)

stopped = False
while not stopped:
    inference_recommender_job = inference_client.describe_inference_recommendations_job(
        JobName=str(advanced_job)
    )
    if inference_recommender_job["Status"] in ["COMPLETED", "STOPPED", "FAILED"]:
        stopped = True
    else:
        print("Inference recommender job in progress")
        time.sleep(600)

if inference_recommender_job["Status"] == "FAILED":
    print("Inference recommender job failed ")
    print("Failed Reason: {}".inference_recommender_job["FailureReason"])
else:
    print("Inference recommender job completed")

In [ ]:
data = [
    {**x["EndpointConfiguration"], **x["ModelConfiguration"], **x["Metrics"]}
    for x in inference_recommender_job["InferenceRecommendations"]
]
df = pd.DataFrame(data)
dropFilter = df.filter(["VariantName"])
df.drop(dropFilter, inplace=True, axis=1)
pd.set_option("max_colwidth", 400)
df.head()